In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [ ]:
# Image => Scale(0, 1) => Transform(-1, 1)

In [11]:
# Datasets and DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transform

transform = transform.Compose([
    transform.ToTensor(), # Automatically convert img into pytorch tensors + Scale
    transform.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Standard values for this particular dataset
])

trainset = CIFAR10(root='./data', train=True, download=True, transform=transform)
# root => where our data is present, train=> if we want train data, download=> in case we dont have dataset it will automatically download it , 
#transform => the num of operation and kind of operation want to do in it
testset = CIFAR10(root='./data', train=False, download=True, transform=transform)

In [12]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [13]:
trainloader = DataLoader(trainset, batch_size =64, shuffle = True)
testloader = DataLoader(testset, batch_size=64)

# Build the CNN

In [18]:
# If the size of img is => 32*32*3 we just flip it => 3*32*32 and this provide to CNN

# CNN => Convolutional layer+ Relu -> MaxPool -> Flattering -> ReLu- >Output

# Kernal/Filtter = 3*3
# padding = 1
# Stride = 1

# FOR MaxPool
# kernal= 2*2
# stride = 1

# IN Pooling layer it half the width and height in each CNN process
# 1st layer - 32*32*32- maxpool- 16*16*32
# 2nd layer- 16*16*64 - maxpool - 8*8*128
# 3rd layer - 8*8*128-  maxpool -4*4*128 

In [44]:
 class CNN(nn.Module):
     def __init__(self):
         super(CNN, self).__init__()

         self.conv_layers = nn.Sequential(
             
             # 1st Layer
             nn.Conv2d(3,32, kernel_size=3, padding=1),#(input, output, kernal, padding)
             nn.ReLU(),
             nn.MaxPool2d(2,2), # Kernal size= 2, stride= 2

             # 2nd Layer    
             nn.Conv2d(32, 64, kernel_size=3, padding=1),
             nn.ReLU(),
             nn.MaxPool2d(2,2),

             #3rd Layer
             nn.Conv2d(64, 128, kernel_size=3, padding=1),
             nn.ReLU(),
             nn.MaxPool2d(2, 2),
               
         )

         self.fc_layers = nn.Sequential(
             nn.Linear(4*4*128, 256), # Input after flattern and then there is Output
             nn.ReLU(),

             nn.Linear(256, 10) # Final 10 classification
         )

     def forward(self, x):
          x = self.conv_layers(x)
          x = x.view(x.size(0), -1) # Flattening
          x = self.fc_layers(x)

          return x

In [45]:
model = CNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# Training CNN

In [47]:
epochs = 10

for epoch in  range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # Forward Propagation
        loss = criterion(output, labels) # Loss Func
        loss.backward() # Back Propagation
        optimizer.step() # Update the weight and bias

        epoch_training_loss += loss.item()

    print(f"epoch={epoch} & loss= {epoch_training_loss/len(trainloader)}")  

epoch=0 & loss= 0.5199821614624595
epoch=1 & loss= 0.42349753894693104
epoch=2 & loss= 0.3380818196936794
epoch=3 & loss= 0.2581033403115809
epoch=4 & loss= 0.20420337911418943
epoch=5 & loss= 0.15886742528766165
epoch=6 & loss= 0.12468846984293379
epoch=7 & loss= 0.11606817094423354
epoch=8 & loss= 0.09924848395985697
epoch=9 & loss= 0.08541411741296082


In [51]:
# Evaluation

correct_labels = 0.0
total_labels =0.0

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _,predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)


print(f"Accuracy = {correct_labels/total_labels *100} ")

Accuracy = 74.11999999999999 
